<a href="https://colab.research.google.com/github/Olacherish/DataScienceClass/blob/main/1_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **CAPSTONE PROJECT: TRANSFER LEARNING WITH PYTORCH**

# **CNN Transfer Learning (Image Classification)**
• Image Classification using ResNet50.

• Dataset: Oxford-IIIT Pet Dataset

## **OBJECTIVES**
Using a pretrained (ResNet50 model) CNN to classify Pet Breeds from the Oxford-IIIT Pet dataset.


# **Concepts Covered**
• CNN feature extraction

• Transfer learning

• Fine-tuning

• Data augmentation

# Dataset
Oxford-IIIT Pet Dataset (37 pet breeds) Load with torchvision.datasets.OxfordIIITPet


# Tasks:
1. Data preprocessing and Augmentation
2. Transfer learning using pretrained ResNet50
3. Train frozen model
4. Fine-tune model
5. Evaluate and compare Accuracies

# Types of CNN Architecture
ResNet50

# Deliverables
• PyTorch notebook with full pipeline.

• Accuracy comparison (frozen vs fine-tuned)

## **SOLUTIONS TO CAPSTONE PROJECT**

## This Capstone project is all about:
- Transfer Learning with Convolutional Neural Networks (CNNs) using PyTorch.
# Goal
- The goal is to classify Pet images into 37 breeds from the Oxford-IIIT Pet dataset.
# Objectives
- Using pretrained model (ResNet50 model) that has already learned features from millions of images instead of training a CNN from the scratch.

# Types of CNN Architecture
ResNet50

# Capstone Project Workflow
Data Preparation
    ↓
Device Configuration
    ↓
Data Transformation and Augmentation
    ↓
Load Pretrained CNN
    ↓
Replace and Train Classifier (Final Layer) Head
    ↓
Freeze Layers
    ↓
Train Fronze Layers
    ↓
Loss function and Optimizer
    ↓
Evaluate Accuracy
    ↓
Fine-tune Entire Model
    ↓
Evaluate Again
    ↓
Compare Results


## **Task 1: Data Preparation and Augmentation**

In [ ]:
# Importing all essential Modules and libraries

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader


import torchvision
from torchvision import datasets, transforms
from torchvision import transforms
from torchvision import models
from torchvision.datasets import OxfordIIITPet

import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
import requests
from io import BytesIO
import time
import os
from tqdm import tqdm
import copy


# **Device Configuration**

In [ ]:
import torch
# Device Configuration
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)
print("Device:", device)

Device: cuda


**Data Transformations and Augmentations**
*   Data Augmentation and Normalization for training
*   Data Augmentation creates new image variations (Augmented versions of the images) to improve regularization and reduce overfitting.

## **Data (images) Preprocessing**

In [ ]:
from torchvision import transforms

# Data Augmentation
# Preprocess images by Training Transforms
train_transforms = transforms.Compose([
    transforms.ToTensor(),              # Convert images to PyTorch tensors
    transforms.Normalize(
      mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Normalize with ImageNet stats for RGB (3)

    transforms.Resize(256),             # Resize images to 256x256
    transforms.CenterCrop(224),         # Crop the center 224x224
    transforms.RandomHorizontalFlip(),  # Randomly flip images horizontally
    transforms.RandomRotation(15),      # Randomly rotate images by +/- 15 degrees
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2 ),

    ])

# Validation of Test Transform
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## **Task 2: Transfer Learning using Pretrained CNN  Architecture (ResNet50) Loading**




In [ ]:
from torchvision import models

# Loading pre-trained ResNet50
# This model will be adapted for classification on the Oxford-IIIT Pet Dataset.

model = torchvision.models.resnet50(weights=True)

from torchvision.models import resnet50, ResNet50_Weights
weights = ResNet50_Weights.DEFAULT

model = resnet50(weights=weights)

In [ ]:
from torch.utils.data import DataLoader
from torchvision.datasets import OxfordIIITPet

train_dataset = OxfordIIITPet(
    root='./data',
    split='trainval',
    transform=train_transforms,
    download=True
)

test_dataset = OxfordIIITPet(
    root='./data',
    split='test',
    transform=val_transform,
    download=True
)

# Create DataLoaders
batch_size = 32 # You can adjust this batch size

train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2       # This is based on system capabilities
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")
print(f"Number of batches in training loader: {len(train_loader)}")
print(f"Number of batches in test loader: {len(test_dataloader)}")

# The number of batches
# images, labels = next(iter(train_loader))
# print(f"Shape of image batch: {images.shape}")
# print(f"Shape of label batch: {labels.shape}")

In [ ]:
import matplotlib.pyplot as plt

from PIL import Image

print("Downloading and preparing dataset...")

train_dataset = datasets.OxfordIIITPet(
    root='./data',
    split='trainval',
    download=True,
    transform=train_transform
)

test_dataset = datasets.OxfordIIITPet(
    root='./data',
    split='test',
    download=True,
    transform=val_transform
)

# Creating DataLoaders
train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("Training Images:", len(train_dataset))
print("Testing Images:", len(test_dataset))

# Exploring Dataset: because images are normalized, they may look different.
# Display Sample Images
images, labels = next(iter(train_dataloader))
fig, axes = plt.subplots(1, 5, figsize=(15,5))

for i in range(5):
    img = images[i].permute(1,2,0)
    img = img.numpy()

    axes[i].imshow(img)
    axes[i].set_title(f"Label: {labels[i]}")
    axes[i].axis("off")

plt.show()

## **Replacing and Training Classifier (Final Layer) Head**

## We replace classifier, by replacing the Final Layer using Original ResNet50 with 1000 Outputs, We need just:37 outputs.

## Replacing the final fully connected layer (classifier)

## The original ResNet50 "fc" layer outputs 1000 classes. We need to replace it with a new layer that matches our specific number of target classes (e.g. 37). "model_resnet.fc.in_features" gives us the input size of the original fc layer.



In [ ]:
import torch

# Device Configuration (re-added against kernel resets/out-of-order execution)
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

# Freeze all parameters in the network
for param in model.parameters():
    param.requires_grad = False

# Get the number of input features for the last layer
num_ftrs = model.fc.in_features

# Replace the last layer (classifier head) with a new one for 37 classes
# The Oxford-IIIT Pet Dataset has 37 breeds.
model.fc = nn.Linear(num_ftrs, 37)

# Move the model to the configured device (CPU or GPU)
model = model.to(device)

print("ResNet50 model with modified classifier head:")
print(model.fc)

ResNet50 model with modified classifier head:
Linear(in_features=2048, out_features=37, bias=True)


## **Loss function and Optimizer**


*   Train only the new classifier Head.

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer: Only the final layer parameters are being optimized instead of optimizing all the parameters again.
optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

# Training parameters
num_epochs = 10       # Number of epochs is not fixed, it can be adjusted

# Loss/Accuracy (training and validation)
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

print("Starting training...")
for epoch in range(num_epochs):
    model.train()             # Set model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0


## **Task 3: Train Frozen model/Layers**

In [ ]:
import copy

def train_model(model,
                dataloaders, # Expects a dictionary: {'train': train_dataloader, 'val': val_dataloader}
                criterion,
                optimizer,
                scheduler,
                num_epochs=5,
                device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")):

    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} (Train)"):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad() # Zero the parameter gradients

        outputs = model(inputs) # Forward pass
        loss = criterion(outputs, labels) # Calculate loss
        loss.backward() # Backward pass
        optimizer.step() # Optimize

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_train_loss = running_loss / len(train_dataset)
    epoch_train_accuracy = 100 * correct_predictions / total_samples
    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_accuracy)



In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=25):
    since = time.time()

    # Model and Criterion are ensured on the correct device
    model.to(device)
    criterion.to(device)

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
            else:
                model.eval()   # Set model to evaluate mode

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            dataloader = train_dataloader if phase == 'train' else test_dataloader
            for inputs, labels in tqdm(dataloader, desc=f'{phase}ing Epoch {epoch}'):
                inputs = inputs.to(device)
                labels = labels.to(device)

                # zero the parameter gradients
                optimizer.zero_grad()

                # forward propagation
                # track history if only in train
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # deep copy the model
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')

    # load best model weights
    model.load_state_dict(best_model_wts)
    return model

# Train the model with frozen layers
model_ft = train_model(model, criterion, optimizer_ft, scheduler, num_epochs=10)
print("Finished training frozen layers.")

Epoch 0/9
----------


training Epoch 0: 100%|██████████| 100/100 [00:50<00:00,  2.00it/s]


train Loss: 7.2518 Acc: 0.0008


valing Epoch 0: 100%|██████████| 100/100 [00:18<00:00,  5.27it/s]


val Loss: 18133.4983 Acc: 0.0000

Epoch 1/9
----------


training Epoch 1:  34%|███▍      | 34/100 [00:18<00:35,  1.88it/s]


KeyboardInterrupt: 

In [ ]:
# Plot loss curves
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curves for MLP on Iris Dataset')
plt.legend()
plt.show()

# Evaluate
model.eval()
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = (predicted == y_test).sum().item() / len(y_test)
    print(f"Test Accuracy: {accuracy:.2f}")
```

## **Task 4: Fine-tune Entire Model**

Now, we will unfreeze all layers of the model and continue training. This allows the pre-trained layers to be adjusted more subtly to the new dataset, potentially leading to higher accuracy.

In [ ]:
# Unfreeze all layers for fine-tuning
for param in model.parameters():
    param.requires_grad = True

print("All layers are now unfrozen for fine-tuning.")

All layers are now unfrozen for fine-tuning.


In [ ]:
# Define a new optimizer for fine-tuning, optimizing all model parameters
# A smaller learning rate is often used for fine-tuning to avoid corrupting pre-trained weights too quickly.
optimizer_ft_unfrozen = optim.SGD(model.parameters(), lr=0.0001, momentum=0.9)

# Define a new learning rate scheduler for fine-tuning
scheduler_ft_unfrozen = lr_scheduler.StepLR(optimizer_ft_unfrozen, step_size=7, gamma=0.1)

print("Optimizer and scheduler for fine-tuning initialized.")

Optimizer and scheduler for fine-tuning initialized.


In [ ]:
# Training the model with unfrozen layers
# Running few more epochs for fine-tuning.
num_epochs_finetune = 20 # adjusting number of epochs
model_finetuned = train_model(
    model,
    criterion,
    optimizer_ft_unfrozen,
    scheduler_ft_unfrozen,
    num_epochs=num_epochs_finetune
)

print("Finished fine-tuning the model.")

Epoch 0/9
----------


training Epoch 0: 100%|██████████| 100/100 [00:49<00:00,  2.03it/s]


train Loss: 7.0732 Acc: 0.0005


valing Epoch 0: 100%|██████████| 100/100 [00:20<00:00,  4.91it/s]


val Loss: 1254.2543 Acc: 0.0000

Epoch 1/9
----------


...

val Loss: 1076.1783 Acc: 0.0011

Epoch 9/9
----------


training Epoch 9: 100%|██████████| 100/100 [00:49<00:00,  2.02it/s]


train Loss: 5.1028 Acc: 0.0481


valing Epoch 9: 100%|██████████| 100/100 [00:19<00:00,  5.24it/s]

val Loss: 998.4843 Acc: 0.0016

Training complete in 11m 26s
Best val Acc: 0.0022
Finished fine-tuning the model.


In [ ]:
from torch.utils.data import DataLoader
from torchvision.datasets import OxfordIIITPet

train_dataset = OxfordIIITPet(
    root='./data',
    split='trainval',
    transform=train_transforms,
    download=True
)

test_dataset = OxfordIIITPet(
    root='./data',
    split='test',
    transform=val_transform,
    download=True
)

# Create DataLoaders
batch_size = 37 # You can adjust this batch size

train_dataloader = DataLoader(
    train_dataset,
    batch_size=37,
    shuffle=True,
    num_workers=2       # This is based on system capabilities
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=37,
    shuffle=False,
    num_workers=2
)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")
print(f"Number of batches in training loader: {len(train_loader)}")
print(f"Number of batches in test loader: {len(test_dataloader)}")

# The number of batches
# images, labels = next(iter(train_loader))
# print(f"Shape of image batch: {images.shape}")
# print(f"Shape of label batch: {labels.shape}")

NameError: name 'train_transforms' is not defined

In [ ]:
from torch.utils.data import DataLoader
from torchvision.datasets import OxfordIIITPet

train_dataset = OxfordIIITPet(
    root='./data',
    split='trainval',
    transform=train_transforms,
    download=True
)

test_dataset = OxfordIIITPet(
    root='./data',
    split='test',
    transform=val_transform,
    download=True
)

# Create DataLoaders
batch_size = 37 # You can adjust this batch size

train_dataloader = DataLoader(
    train_dataset,
    batch_size=37,
    shuffle=True,
    num_workers=2       # This is based on system capabilities
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=37,
    shuffle=False,
    num_workers=2
)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")
print(f"Number of batches in training loader: {len(train_dataloader)}")
print(f"Number of batches in test loader: {len(test_dataloader)}")

# The number of batches
# images, labels = next(iter(train_loader))
# print(f"Shape of image batch: {images.shape}")
# print(f"Shape of label batch: {labels.shape}")

Number of training samples: 3680
Number of test samples: 3669
Number of batches in training loader: 100
Number of batches in test loader: 100
